# ML-03: Frame Your Lane as an ML Task

Week 2. Last week I picked Lane 2 (Refresh / Content Opportunity Scoring) and argued that the real problem is ordering pages for a review team with limited time. This notebook turns that into something a model can be trained and judged on: the task type, the target, the metric, and the unit of analysis as an actual dataframe. All numbers come from the starter slice; where the warehouse will change things, I say so.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

def find_repo_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return folder
    raise FileNotFoundError("Run this from inside a clone of the repo.")

ROOT = find_repo_root(Path.cwd().resolve())
df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
K = 50  # review capacity from week 1
print(f"{len(df):,} pages, {df['client_id'].nunique()} clients")

30,000 pages, 32 clients


## 1. My lane as an ML task (type)

It's a ranking task, built on top of a binary classifier.

The editor never sees a yes/no answer. They see a list, sorted, and they work down it until they run out of time. So the thing I actually ship is an ordering, and the only part of it that matters is the top of the list. Underneath, the cleanest way to get that ordering is pointwise: train a classifier that outputs P(page declines in the next window), then sort by that probability (mixed with demand, see section 3). I'm not going to try a learning-to-rank setup with pairwise losses. With one label per page and no relevance grades, it adds complexity without giving the model anything new to learn from.

It isn't clustering, because clusters have no "first". And it isn't plain regression on the size of the drop. The drop size is very heavy-tailed, and a regression would spend most of its effort on a handful of huge pages.

The cell below shows why ranking needs its own yardstick. If I flagged every page as declining, I'd be right 54% of the time. That number is the floor any ordering has to clear at the top of the list, not something to report as accuracy.

In [2]:
label = df["trend_direction"].eq("down")
base_rate = label.mean()
print(f"starter proxy base rate: {base_rate:.1%}")
print(f"a random top-{K} would contain about {base_rate * K:.0f} positives")
print(f"'flag everything' accuracy: {base_rate:.1%}   (useless, ranks nothing)")

starter proxy base rate: 54.2%
a random top-50 would contain about 27 positives
'flag everything' accuracy: 54.2%   (useless, ranks nothing)


## 2. Target or proxy

What I'd like to predict: whether a page that has real search demand today will lose a meaningful share of it over the next 30 days, beyond what its own site is losing.

What the starter gives me is a proxy, and a weak one. `trend_direction == "down"` is a defined rule (last 30 days more than 20% below the previous 30), and both windows sit inside the same 90 days the page features describe. Week 1 showed it can be rebuilt from the current columns on every row. So it's a label from a rule, measured at the same moment as the features. A model trained on it learns to recognise a drop that already happened.

The target I'll build on the warehouse daily table, with decision date *t* chosen per client inside their tracked history:

- Features: only data from the 90 days before *t*.
- Eligibility: at least 100 impressions in the 30 days before *t*, page at least 90 days old, client's search tracking active across both windows.
- Label = 1 when impressions in [*t*, *t*+30) fall more than 20% below [*t*-30, *t*), and the page's change is at least 20 points worse than its client's median change over the same window.
- Feature and label windows never share a day; the leakage notebook in week 3 checks that row by row.

The volume floor and the client adjustment are both there to strip out things that aren't page decay. The cell below tests them on the starter proxy to see how much of the label they remove. It's the same rule applied to current windows, so it only shows the size of the effect, not the final label.

In [3]:
d = df.copy()
d["eligible"] = d["impressions_prev_30d"].ge(100)
d["pct_change"] = np.where(
    d["impressions_prev_30d"] > 0,
    (d["impressions_last_30d"] - d["impressions_prev_30d"]) / d["impressions_prev_30d"].where(d["impressions_prev_30d"] > 0) * 100,
    np.nan,
)
client_median = d[d["eligible"]].groupby("client_id")["pct_change"].median()
d["vs_client"] = d["pct_change"] - d["client_id"].map(client_median)

raw = d["trend_direction"].eq("down")
with_floor = raw & d["eligible"]
client_relative = with_floor & d["vs_client"].lt(-20)

steps = pd.DataFrame({
    "label version": ["starter proxy (rule)", "+ volume floor (prev 30d >= 100)", "+ worse than own client by 20 pts"],
    "positives": [raw.sum(), with_floor.sum(), client_relative.sum()],
    "rate within its population": [raw.mean(), with_floor.sum() / d["eligible"].sum(), client_relative.sum() / d["eligible"].sum()],
})
steps["rate within its population"] = steps["rate within its population"].map("{:.1%}".format)
print(f"pages below the volume floor: {(~d['eligible']).sum():,}")
print(f"tiny pages (1-10 prev impressions) still tagged down: {(raw & d['impressions_prev_30d'].between(1, 10)).sum():,}")
steps

pages below the volume floor: 11,990
tiny pages (1-10 prev impressions) still tagged down: 1,742


,label version,positives,rate within its population
0,starter proxy (rule),16262,54.2%
1,+ volume floor (prev 30d >= 100),11090,61.6%
2,+ worse than own client by 20 pts,4495,25.0%


The volume floor removes about 5,200 positives, most of them small-number wobble: a page going from 6 impressions to 4 counts as "down" under the starter rule, and 1,742 pages with 10 or fewer impressions carry the tag. The client adjustment matters much more. Among eligible pages the positive rate falls from 61.6% to 25.0%. Most of what the starter calls page decline moves with the rest of the client's site. That's the part an editor can't fix by refreshing one page, and it's what I want the label to leave out.

## 3. Success metric

Primary metric: precision@50 on held-out clients. Of the 50 pages the queue puts first, how many really declined (by the section 2 label)? It matches the decision exactly: K is the review capacity, and the editor only ever sees the top.

It needs a companion, because precision@50 counts every page the same. Week 1 showed that 10% of pages hold 70% of impressions, so a queue could hit a decent precision by finding lots of small declines and miss the few that matter. Second metric: demand captured@50, the impressions lost by the true positives in my top 50, divided by the impressions lost by the best possible top 50. 1.0 means the queue found the costliest declines; 0 means it found none.

What counts as good. I'm fixing these before training anything:

- Precision@50 clearly above the base rate (at least 1.5×), and above the transparent rule baseline in most client-holdout folds, not just on average.
- Demand captured@50 at least as high as the rule baseline's. A model that wins on precision by picking small pages loses.
- Recall and average precision get reported, but they don't decide anything.

The cell defines both metrics once, so later notebooks import the same logic, and runs them on three orderings that use no model at all. They're scored against the starter proxy, so they're reference points, not results.

In [4]:
def precision_at_k(score, y, k=K):
    top = score.sort_values(ascending=False).index[:k]
    return y.loc[top].mean()

def demand_captured_at_k(score, y, lost, k=K):
    top = score.sort_values(ascending=False).index[:k]
    found = (lost.loc[top] * y.loc[top]).sum()
    best = (lost * y).sort_values(ascending=False).iloc[:k].sum()
    return found / best

lost = (df["impressions_prev_30d"] - df["impressions_last_30d"]).clip(lower=0)
rng = np.random.default_rng(42)
orderings = {
    "random": pd.Series(rng.random(len(df)), index=df.index),
    "biggest pages first (impressions_90d)": df["impressions_90d"].astype(float),
    "stalest first (days_since_last_update)": df["days_since_last_update"].astype(float),
}
rows = [
    {"ordering": name,
     f"precision@{K}": precision_at_k(s, label),
     f"demand captured@{K}": demand_captured_at_k(s, label, lost)}
    for name, s in orderings.items()
]
pd.DataFrame(rows).round(3)

,ordering,precision@50,demand captured@50
0,random,0.44,0.003
1,biggest pages first (impressions_90d),0.42,0.471
2,stalest first (days_since_last_update),0.52,0.000


Random lands at 0.44, within sampling noise of the base rate (a 50-page sample swings by roughly 7 points either way), and captures almost none of the lost demand. Sorting by size captures about half the possible lost demand but has the worst precision, because the biggest pages are a little less likely to be tagged down. Staleness sits at base-rate precision and captures none of it. None of these is a baseline I'd defend; the real rule baseline comes in week 4. They do show that the two metrics disagree, which is exactly why I'm keeping both.

## 4. The unit of analysis, as a real dataframe

In the starter slice, one row = one page, described by trailing 90-day totals at a single export date. On the warehouse, one row = one page at one decision date, so the same page can appear several times with different windows. That's the version the capstone will use, and it's why validation has to split by client (and later by time), never by row.

Below is the starter frame with every column assigned a role before any modeling. The roles carry into the week 3 data contract.

In [5]:
assert df["content_id"].is_unique, "grain broken: duplicate pages"

roles = {
    "id (grouping and splits only)": ["content_id", "client_id"],
    "label source (never a feature)": ["trend_direction", "trend_pct"],
    "overlaps the label windows (use only in a prior window)": [
        "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
        "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    ],
    "excluded (generation metadata, not a page signal)": ["provider_used", "model_used"],
}
assigned = {c for cols in roles.values() for c in cols}
roles["candidate feature"] = [c for c in df.columns if c not in assigned]

role_table = pd.DataFrame(
    [{"role": r, "n": len(cols), "columns": ", ".join(cols)} for r, cols in roles.items()]
)
print(f"columns assigned a role: {sum(role_table['n'])} of {df.shape[1]}")
role_table[["role", "n"]]

columns assigned a role: 44 of 44


,role,n
0,id (grouping and splits only),2
1,label source (never a feature),2
2,overlaps the label windows (use only in a prio...,6
3,"excluded (generation metadata, not a page signal)",2
4,candidate feature,32


In [6]:
show = ["content_id", "client_id", "content_type", "content_age_days", "days_since_last_update",
        "impressions_90d", "avg_position", "ctr", "sessions_90d", "engagement_rate"]
d.loc[d["eligible"], show].sample(5, random_state=7)

,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,sessions_90d,engagement_rate
27870,content_a13924fb5eb3,client_4e07408562,keyword article,277,25,5553,13.6,0.59,17,0.0
3096,content_ffca1716a06b,client_19581e27de,keyword article,463,20,364,1.0,0.00,7,0.0
16165,content_05fb66f99f0d,client_3fdba35f04,keyword article,333,104,1225,17.0,0.08,26,0.0
16516,content_8507ff4057e7,client_19581e27de,keyword article,257,104,1040,61.0,0.00,7,0.0
3629,content_5776fb1c426f,client_19581e27de,keyword article,139,104,322,5.9,0.00,1,0.0


In [7]:
print(f"eligible rows under the volume floor: {d['eligible'].sum():,} of {len(d):,}")
print(d.loc[d["eligible"]].groupby("client_id").size().describe().round(0).to_string())

eligible rows under the volume floor: 18,010 of 30,000
count      30.0
mean      600.0
std      1218.0
min         1.0
25%        18.0
50%       150.0
75%       532.0
max      5690.0


The eligible population is about 18,000 pages, still very uneven across clients: some have a handful of eligible pages, one has thousands. With a client-holdout split, a single big client landing in the test fold can swing precision@50 a lot, so I'll report the metric per fold, not only the mean. IDs shown above are pseudonyms; no client names or URLs exist in this data.

## 5. Why ML beats a fixed rule here

Two things in the data make a hand-written rule a poor fit.

First, no single signal separates the classes well. The best one alone (page age) gets an AUC around 0.59, barely better than a coin. Several others sit near 0.5. The starter random forest reached 0.75 by combining them, so the information is spread thin across many columns rather than sitting in one.

Second, the signals interact, and the direction of an effect flips depending on another signal. The pivot below shows the decline rate by position tier and age tier, only for visible pages, and only for cells with at least 100 pages. For pages older than a year, the decline rate falls steadily the further down the results a page sits, from 0.50 on page 1 to 0.20 for deep positions. For pages aged 91 to 180 days it moves the other way between page 1 and pages 3 to 5 (0.62 up to 0.76). An if-statement would have to encode every one of those exceptions by hand, and they'd shift between clients and months.

A caveat I want written down: part of this signal comes from columns like `impressions_90d` and `days_with_impressions` that share days with the starter label windows. On the warehouse, with a clean future label, these AUCs will probably drop. The case for ML here is that the pattern is spread out and conditional; whether it's strong enough to be useful has to be earned in weeks 5 and 6.

In [8]:
def auc(x, y):
    x = x.fillna(x.median())
    r = x.rank()
    n1, n0 = y.sum(), len(y) - y.sum()
    a = (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
    return max(a, 1 - a)

signals = ["content_age_days", "impressions_90d", "days_with_impressions", "search_volume",
           "word_count", "avg_position", "days_since_last_update", "sessions_90d", "ctr", "engagement_rate"]
single = pd.Series({c: auc(df[c].astype(float), label) for c in signals}).sort_values(ascending=False)
print("single-signal AUC vs starter proxy (0.5 = no separation)")
print(single.round(3).to_string())

single-signal AUC vs starter proxy (0.5 = no separation)
content_age_days          0.591
impressions_90d           0.584
days_with_impressions     0.579
search_volume             0.560
word_count                0.534
avg_position              0.530
days_since_last_update    0.527
sessions_90d              0.519
ctr                       0.515
engagement_rate           0.504


In [9]:
visible = df[df["impressions_90d"].ge(500) & df["position_tier"].ne("no_data")]
rate = visible.pivot_table(index="position_tier", columns="age_tier",
                           values="trend_direction", aggfunc=lambda s: s.eq("down").mean())
count = visible.pivot_table(index="position_tier", columns="age_tier",
                            values="content_id", aggfunc="size")
order_rows = ["top_3", "page_1", "striking", "page_3_5", "deep"]
order_cols = ["91-180", "181-365", "365+"]
rate.loc[order_rows, order_cols].where(count.loc[order_rows, order_cols] >= 100).round(2)

age_tier,91-180,181-365,365+
position_tier,,,
top_3,0.79,0.70,NaN
page_1,0.62,0.57,0.50
striking,0.70,0.61,0.47
page_3_5,0.76,0.62,0.34
deep,NaN,0.39,0.20


## Self-check

- [x] Every section above is filled, markdown thinking and the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere (pseudonymized ids and aggregates only)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`